Q-2

Case study

A text-classification service receives messages from web forms and message queues. Web forms provide strings, while queue consumers sometimes provide bytes or bytearray objects. The NLP pipeline needs consistent tokens before further processing. Your function must standardise valid messages without silently accepting undecodable binary content.

Coding task

Required function or decorator

INTERFACE

prepare_messages(payloads, min_length=3)


•	Accept strings directly. Decode bytes and bytearray using UTF-8. Reject unsupported types and binary payloads that cannot be decoded.

•	Lowercase accepted text, split on whitespace, and remove only the characters . , ! ? ; : from both ends of each token. Keep tokens of at least min_length characters, preserving repetition and order.

•	Return one token list per accepted payload, including empty lists; rejected payload positions; and a sorted list of unique retained tokens across all accepted payloads.

•	Use a list comprehension for at least one token transformation or filtering operation and a set comprehension to construct the vocabulary.

Sample input

PYTHON


    payloads = [
    "  AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",]
    prepare_messages(payloads, min_length=3)


EXPECTED RESULT
  
    {
    "token_lists": [
        ["helps", "teams"],
        ["build", "reliable", "agents"],
        ["helps"],
    ],
    "vocabulary": ["agents", "build", "helps", "reliable", "teams"],
    "rejected_indices": [3, 4],
    }

Constraints and edge cases

•	The outer input is a list. min_length must be a positive integer excluding Booleans; otherwise raise ValueError.

•	Empty or whitespace-only text is accepted and produces an empty token list.

•	Internal punctuation is unchanged: "agent-based" remains "agent-based".

•	Do not modify supplied bytearray objects. Empty input produces three empty output lists.

Student deliverables
Working code, visible sample output, at least three additional meaningful test cases, and a brief explanation of your approach.


In [28]:
def prepare_messages(payloads,min_length=3):
  if type(min_length)!= int or min_length<=0:
    raise ValueError("Len must be positive integer")

  rejected_indices=[]
  token_list=[]

  punctuations = ".,!?;:"

  for i, payload in enumerate(payloads):
    if isinstance(payload,str):
      text= payload
    elif isinstance(payload,bytes):
      try:
        text=payload.decode("utf-8")
      except UnicodeDecodeError:
        rejected_indices.append(i)
        continue
    elif isinstance(payload,bytearray):
      try:
        text= bytes(payload).decode("utf-8")
      except UnicodeDecodeError:
        rejected_indices.append(i)
        continue
    else:
      rejected_indices.append(i)
      continue

    words= text.lower().split()
    tokens=[
        w.strip(punctuations)
        for w in words
        if len(w.strip(punctuations))>=min_length
    ]
    token_list.append(tokens)

  vocabulary = sorted({
        token
        for tokens in token_list
        for token in tokens
  })

  return {
        "token_lists": token_list,
        "vocabulary": vocabulary,
        "rejected_indices": rejected_indices
  }



In [16]:
payloads = [
    " AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",
]

In [19]:
from pprint import pprint

result = prepare_messages(payloads)

pprint(result)

{'rejected_indices': [3, 4],
 'token_lists': [['helps', 'teams'],
                 ['build', 'reliable', 'agents'],
                 ['helps']],
 'vocabulary': ['agents', 'build', 'helps', 'reliable', 'teams']}


In [29]:
test1 = [
    "",
    "   ",
    "AI is great!"
]

pprint(prepare_messages(test1))

{'rejected_indices': [],
 'token_lists': [[], [], ['great']],
 'vocabulary': ['great']}


In [30]:
test2 = [
    "agent-based systems!",
    "AI, AI; AI?"
]

pprint(prepare_messages(test2))

{'rejected_indices': [],
 'token_lists': [['agent-based', 'systems'], []],
 'vocabulary': ['agent-based', 'systems']}


In [31]:
test3 = [
    "Hello world",
    None,
    123,
    b"\xff"
]

pprint(prepare_messages(test3))

{'rejected_indices': [1, 2, 3],
 'token_lists': [['hello', 'world']],
 'vocabulary': ['hello', 'world']}


* The function first checks that `min_length` is a positive integer and then processes each payload one by one. Strings are accepted directly, while bytes and bytearray objects are decoded using UTF-8.
* Unsupported types and undecodable binary data are rejected and their positions are stored in `rejected_indices`.

* For valid text, the function converts it to lowercase, splits it into tokens, removes the specified punctuation from the beginning and end of each token, and keeps only tokens that meet the minimum length.
* A list comprehension is used for token processing, while a set comprehension is used to create the unique vocabulary.
* Finally, the vocabulary is sorted and all required results are returned.